In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/wsb_sentiment_top_tickers_with_timestamp.csv",
                 parse_dates=["timestamp"])

df.head()

/tmp/ipykernel_21491/790632648.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


,text,date,ticker,num_tickers,ticker_candidates,caps_tokens,tickers_base,num_tickers_base,dominant_ticker,dominant_count,sentiment_score,sentiment_3,sentiment_5,sentiment_norm,timestamp
0,Math Professor Scott Steiner says the numbers ...,2021-01-28,GME,1,[],[],[],0,GME,492,-0.6249,negative,very negative.,0.18755,2021-01-28 21:32:10
1,Exit the system The CEO of NASDAQ pushed to ha...,2021-01-28,GME,1,"['CEO', 'SEC', 'GME', 'I']","['CEO', 'SEC', 'GME', 'I']",['GME'],1,GME,492,-0.1644,negative,negative,0.41780,2021-01-28 21:30:35
2,NEW SEC FILING FOR GME! CAN SOMEONE LESS RETAR...,2021-01-28,GME,1,"['NEW', 'SEC', 'FOR', 'GME', 'CAN', 'LESS', 'T...","['NEW', 'SEC', 'FOR', 'GME', 'CAN', 'LESS', 'T...",['GME'],1,GME,492,-0.3397,negative,negative,0.33015,2021-01-28 21:28:57
3,"Not to distract from GME, just thought our AMC...",2021-01-28,AMC,2,"['GME', 'AMC']","['GME', 'AMC']","['AMC', 'GME']",2,GME,492,0.2235,positive,positive,0.61175,2021-01-28 21:26:56
4,"Not to distract from GME, just thought our AMC...",2021-01-28,GME,2,"['GME', 'AMC']","['GME', 'AMC']","['AMC', 'GME']",2,GME,492,0.2235,positive,positive,0.61175,2021-01-28 21:26:56


In [3]:
from datetime import time, timedelta
import pytz

# timezone-aware timestamps in US/Eastern (market time)
df["timestamp_utc"] = df["timestamp"].dt.tz_localize("UTC")
df["timestamp_et"] = df["timestamp_utc"].dt.tz_convert("US/Eastern")

# next business day (Mon–Fri), ignoring holidays for now
def next_business_day(d):
    while d.weekday() >= 5:  # 5 = Saturday, 6 = Sunday
        d += timedelta(days=1)
    return d

# Assign a "trading_day" per post based on market hours
MARKET_OPEN = time(9, 30)
MARKET_CLOSE = time(16, 0)

def assign_trading_day(ts_et):
    """
    Map a post timestamp in US/Eastern to the trading day it should be
    associated with for price movement analysis.

    Rules:
      - If weekend: use the next business day.
      - If after 4:00 pm: sentiment is for the *next* trading day.
      - If before 9:30 am: treat as pre-market for the *same* trading day.
      - If between 9:30 am and 4:00 pm: same trading day.
    """
    d = ts_et.date()
    t = ts_et.time()

    # Weekend, push to next business day
    if ts_et.weekday() >= 5:
        return next_business_day(d)

    # Weekday
    if t >= MARKET_CLOSE:
        # After close -> next trading day
        return next_business_day(d)
    else:
        # Pre-market or during session
        return d

df["trading_day"] = df["timestamp_et"].apply(assign_trading_day)
df[["timestamp_et", "trading_day", "ticker", "sentiment_score"]].head()


,timestamp_et,trading_day,ticker,sentiment_score
0,2021-01-28 16:32:10-05:00,2021-01-28,GME,-0.6249
1,2021-01-28 16:30:35-05:00,2021-01-28,GME,-0.1644
2,2021-01-28 16:28:57-05:00,2021-01-28,GME,-0.3397
3,2021-01-28 16:26:56-05:00,2021-01-28,AMC,0.2235
4,2021-01-28 16:26:56-05:00,2021-01-28,GME,0.2235


In [5]:
df.columns.tolist()
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65168 entries, 0 to 65167
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype                     
---  ------             --------------  -----                     
 0   text               65168 non-null  object                    
 1   date               65168 non-null  object                    
 2   ticker             65168 non-null  object                    
 3   num_tickers        65168 non-null  int64                     
 4   ticker_candidates  65168 non-null  object                    
 5   caps_tokens        65168 non-null  object                    
 6   tickers_base       65168 non-null  object                    
 7   num_tickers_base   65168 non-null  int64                     
 8   dominant_ticker    65168 non-null  object                    
 9   dominant_count     65168 non-null  int64                     
 10  sentiment_score    65168 non-null  float64                   
 11  sentiment_3    

In [7]:
print("Date range:", df['date'].min(), "→", df['date'].max())


Date range: 2020-09-29 → 2021-08-16


In [10]:
df['trading_day'] = pd.to_datetime(df['trading_day']).dt.date
df['trading_day'].min(), df['trading_day'].max()

(datetime.date(2020, 9, 28), datetime.date(2021, 8, 16))

In [11]:
import yfinance as yf

tickers = sorted(df['ticker'].unique())
tickers

['AAPL',
 'AMC',
 'AMD',
 'BB',
 'CLOV',
 'DD',
 'GME',
 'IPO',
 'NOK',
 'PLTR',
 'RH',
 'RKT',
 'SNDL',
 'SPCE',
 'SPY',
 'TSLA',
 'UWMC']

In [12]:
start = pd.to_datetime(df['trading_day']).min() - pd.Timedelta(days=10)
end   = pd.to_datetime(df['trading_day']).max() + pd.Timedelta(days=10)

start, end

(Timestamp('2020-09-18 00:00:00'), Timestamp('2021-08-26 00:00:00'))

In [13]:
prices_wide = yf.download(
    tickers,
    start=start,
    end=end
)['Close']   # shape: (dates, tickers)

prices_wide.head()

/tmp/ipykernel_21491/1023146741.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  prices_wide = yf.download(
[*********************100%***********************]  17 of 17 completed


Ticker,AAPL,AMC,AMD,BB,CLOV,DD,GME,IPO,NOK,PLTR,RH,RKT,SNDL,SPCE,SPY,TSLA,UWMC
Date,,,,,,,,,,,,,,,,,
2020-09-18,103.824539,56.700001,74.930000,4.86,12.56,22.430807,2.3675,49.500519,3.673928,NaN,378.420013,18.711681,2.64,344.200012,308.252014,147.383331,7.818911
2020-09-21,106.973083,53.200001,77.940002,4.76,11.93,21.254219,2.1875,49.996719,3.557152,NaN,371.970001,17.997494,2.36,329.600006,304.821320,149.796661,7.776261
2020-09-22,108.654236,52.099998,77.699997,4.94,12.00,21.095327,2.6400,50.949421,3.575118,NaN,371.959991,18.123526,2.27,329.399994,307.925720,141.410004,7.683856
2020-09-23,104.096619,47.799999,74.730003,4.80,11.66,20.569450,2.5100,50.234890,3.485291,NaN,352.660004,17.056448,2.09,314.600006,300.784637,126.786667,7.427965
2020-09-24,105.165565,46.099998,75.820000,4.75,11.65,20.542969,2.2850,48.984474,3.449359,NaN,351.200012,16.854795,1.88,318.399994,301.586365,129.263336,7.250263


In [14]:
price_rows = []

for t in tickers:
    if t not in prices_wide.columns:
        print(f"⚠️ No price data for {t}, skipping.")
        continue
    
    s = prices_wide[t].dropna()  # series indexed by Timestamp
    tmp = pd.DataFrame({
        'trading_day': s.index.date,   # convert index to plain date
        'ticker': t,
        'close': s.values
    })
    price_rows.append(tmp)

price_df = pd.concat(price_rows, ignore_index=True)
price_df.head()

,trading_day,ticker,close
0,2020-09-18,AAPL,103.824539
1,2020-09-21,AAPL,106.973083
2,2020-09-22,AAPL,108.654236
3,2020-09-23,AAPL,104.096619
4,2020-09-24,AAPL,105.165565


In [15]:
df_with_price = df.merge(
    price_df,
    on=['ticker', 'trading_day'],
    how='left'
)

df_with_price[['ticker', 'trading_day', 'close']].head()

,ticker,trading_day,close
0,GME,2021-01-28,48.400002
1,GME,2021-01-28,48.400002
2,GME,2021-01-28,48.400002
3,AMC,2021-01-28,86.300003
4,GME,2021-01-28,48.400002


In [16]:
total = len(df_with_price)
matched = df_with_price['close'].notna().sum()
coverage = matched / total * 100

total, matched, round(coverage, 2)

(65168, 63896, 98.05)

In [17]:
price_df.head()
# columns: ['trading_day', 'ticker', 'close']

,trading_day,ticker,close
0,2020-09-18,AAPL,103.824539
1,2020-09-21,AAPL,106.973083
2,2020-09-22,AAPL,108.654236
3,2020-09-23,AAPL,104.096619
4,2020-09-24,AAPL,105.165565


In [18]:
# Make sure trading_day is datetime.date in price_df too
price_df['trading_day'] = pd.to_datetime(price_df['trading_day']).dt.date

returns_rows = []

for t, sub in price_df.groupby('ticker'):
    sub = sub.sort_values('trading_day').copy()
    
    # Backward returns (how much it moved in the past up to this day)
    # Ex: ret_back_1d at D0 = (close_D0 - close_D-1) / close_D-1
    sub['ret_back_1d'] = sub['close'].pct_change(1)
    sub['ret_back_3d'] = sub['close'].pct_change(3)
    sub['ret_back_7d'] = sub['close'].pct_change(7)

    # Forward returns (how much it moves after this day)
    # Ex: ret_fwd_1d at D0 = (close_D1 - close_D0) / close_D0
    sub['ret_fwd_1d'] = sub['close'].pct_change(1).shift(-1)
    sub['ret_fwd_3d'] = sub['close'].pct_change(3).shift(-3)
    sub['ret_fwd_7d'] = sub['close'].pct_change(7).shift(-7)

    returns_rows.append(sub)

returns_df = pd.concat(returns_rows).reset_index(drop=True)
returns_df.head()

,trading_day,ticker,close,ret_back_1d,ret_back_3d,ret_back_7d,ret_fwd_1d,ret_fwd_3d,ret_fwd_7d
0,2020-09-18,AAPL,103.824539,NaN,NaN,NaN,0.030326,0.002621,0.067858
1,2020-09-21,AAPL,106.973083,0.030326,NaN,NaN,0.015716,-0.016897,0.052053
2,2020-09-22,AAPL,108.654236,0.015716,NaN,NaN,-0.041946,0.004204,0.044540
3,2020-09-23,AAPL,104.096619,-0.041946,0.002621,NaN,0.010269,0.073189,0.055078
4,2020-09-24,AAPL,105.165565,0.010269,-0.016897,NaN,0.037516,0.054242,0.076511


In [20]:
df_returns = df_with_price.merge(
    returns_df[
        ['ticker', 'trading_day',
         'ret_back_1d', 'ret_back_3d', 'ret_back_7d',
         'ret_fwd_1d',  'ret_fwd_3d',  'ret_fwd_7d']
    ],
    on=['ticker', 'trading_day'],
    how='left'
)
df_returns[
    ['ticker', 'trading_day', 'close', 'sentiment_norm',
     'ret_back_1d', 'ret_fwd_1d',
     'ret_back_3d', 'ret_fwd_3d',
     'ret_back_7d', 'ret_fwd_7d']
].head()


,ticker,trading_day,close,sentiment_norm,ret_back_1d,ret_fwd_1d,ret_back_3d,ret_fwd_3d,ret_back_7d,ret_fwd_7d
0,GME,2021-01-28,48.400002,0.18755,-0.442894,0.678719,1.521162,-0.535124,3.918699,-0.690083
1,GME,2021-01-28,48.400002,0.41780,-0.442894,0.678719,1.521162,-0.535124,3.918699,-0.690083
2,GME,2021-01-28,48.400002,0.33015,-0.442894,0.678719,1.521162,-0.535124,3.918699,-0.690083
3,AMC,2021-01-28,86.300003,0.61175,-0.566332,0.536501,0.952489,-0.093859,1.820262,-0.283893
4,GME,2021-01-28,48.400002,0.61175,-0.442894,0.678719,1.521162,-0.535124,3.918699,-0.690083


In [21]:
output_path = "../data/processed/wsb_sentiment_with_returns.csv"
df_returns.to_csv(output_path, index=False)
output_path

'../data/processed/wsb_sentiment_with_returns.csv'

In [22]:
import os
os.path.getsize(output_path) / 1e6   # file size in MB


157.658959

In [23]:
df.columns.tolist()

['text',
 'date',
 'ticker',
 'num_tickers',
 'ticker_candidates',
 'caps_tokens',
 'tickers_base',
 'num_tickers_base',
 'dominant_ticker',
 'dominant_count',
 'sentiment_score',
 'sentiment_3',
 'sentiment_5',
 'sentiment_norm',
 'timestamp',
 'timestamp_utc',
 'timestamp_et',
 'trading_day']

In [24]:
df_returns.columns.tolist()


['text',
 'date',
 'ticker',
 'num_tickers',
 'ticker_candidates',
 'caps_tokens',
 'tickers_base',
 'num_tickers_base',
 'dominant_ticker',
 'dominant_count',
 'sentiment_score',
 'sentiment_3',
 'sentiment_5',
 'sentiment_norm',
 'timestamp',
 'timestamp_utc',
 'timestamp_et',
 'trading_day',
 'close',
 'ret_back_1d',
 'ret_back_3d',
 'ret_back_7d',
 'ret_fwd_1d',
 'ret_fwd_3d',
 'ret_fwd_7d']

In [25]:
# Select only the columns we actually care about for modeling & analysis

cols_keep = [
    'text',
    'ticker',
    'trading_day',
    'timestamp_et',
    'close',
    'sentiment_score',
    'sentiment_norm',
    # backward returns
    'ret_back_1d',
    'ret_back_3d',
    'ret_back_7d',
    # forward returns
    'ret_fwd_1d',
    'ret_fwd_3d',
    'ret_fwd_7d'
]

# distilled modeling dataframe with only selected columns
df_model = df_returns[cols_keep].copy()

df_model = df_model.dropna(subset=['ret_fwd_1d', 'ret_fwd_3d', 'ret_fwd_7d'])

# Save to processed folder
output_path = "../data/processed/wsb_model_ready.csv"
df_model.to_csv(output_path, index=False)

output_path, df_model.shape


('../data/processed/wsb_model_ready.csv', (63896, 13))

In [26]:
df_model.head()
df_model.columns


Index(['text', 'ticker', 'trading_day', 'timestamp_et', 'close',
       'sentiment_score', 'sentiment_norm', 'ret_back_1d', 'ret_back_3d',
       'ret_back_7d', 'ret_fwd_1d', 'ret_fwd_3d', 'ret_fwd_7d'],
      dtype='object')